In [0]:
dbutils.widgets.text("catalog", "pa_clock_dev", "Catalog")
dbutils.widgets.dropdown("send_email", "true", ["true", "false"], "Send email")
catalog = dbutils.widgets.get("catalog")
SEND_EMAIL = dbutils.widgets.get("send_email") == "true"
SCOPE = "pa_clock_scope"
spark.sql(f"USE CATALOG `{catalog}`")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS pa_clock_dev.gold.gold_pa_alerts (
  alert_id STRING, case_id STRING, risk_band STRING, urgency STRING, line_of_business STRING,
  hours_remaining DOUBLE, due_ts TIMESTAMP, raised_at TIMESTAMP,
  notified BOOLEAN, notified_at TIMESTAMP)
""")

# Raise each (case, risk band) once
spark.sql("""
MERGE INTO pa_clock_dev.gold.gold_pa_alerts t
USING (
  SELECT concat(case_id, '|', risk_band) AS alert_id, case_id, risk_band, urgency, line_of_business,
         CAST(hours_remaining AS DOUBLE) AS hours_remaining, due_ts, current_timestamp() AS raised_at
  FROM pa_clock_dev.gold.gold_pa_open_cases_at_risk
  WHERE risk_band IN ('WARNING', 'CRITICAL', 'BREACHED')
) s
ON t.alert_id = s.alert_id
WHEN NOT MATCHED THEN INSERT
  (alert_id, case_id, risk_band, urgency, line_of_business, hours_remaining, due_ts, raised_at, notified, notified_at)
  VALUES (s.alert_id, s.case_id, s.risk_band, s.urgency, s.line_of_business,
          s.hours_remaining, s.due_ts, s.raised_at, false, NULL)
""")

display(spark.sql("""
  SELECT risk_band, count(*) AS alerts, sum(CASE WHEN notified THEN 0 ELSE 1 END) AS pending
  FROM pa_clock_dev.gold.gold_pa_alerts GROUP BY 1 ORDER BY 1"""))

In [0]:
import smtplib
from email.message import EmailMessage

new = spark.sql("""
  SELECT case_id, risk_band, urgency, line_of_business, hours_remaining, due_ts
  FROM pa_clock_dev.gold.gold_pa_alerts WHERE notified = false
  ORDER BY CASE risk_band WHEN 'BREACHED' THEN 1 WHEN 'CRITICAL' THEN 2 ELSE 3 END, hours_remaining
""").toPandas()

if new.empty:
    print("No new alerts.")
else:
    summary = ", ".join(f"{k}: {v}" for k, v in new["risk_band"].value_counts().to_dict().items())
    body = (f"PA Clock: {len(new)} new alert(s)\n{summary}\n"
            "(negative hours_remaining = already past the deadline)\n\n"
            "Most urgent (top 15):\n" + new.head(15).to_string(index=False))
    print(body)
    if SEND_EMAIL:
        user = dbutils.secrets.get(SCOPE, "smtp-user")
        msg = EmailMessage()
        msg["Subject"] = f"[PA Clock] {len(new)} new PA alerts ({summary})"
        msg["From"] = user
        msg["To"] = dbutils.secrets.get(SCOPE, "alert-to")
        msg.set_content(body)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=30) as s:
            s.login(user, dbutils.secrets.get(SCOPE, "smtp-app-password"))
            s.send_message(msg)
        spark.sql("UPDATE pa_clock_dev.gold.gold_pa_alerts SET notified = true, notified_at = current_timestamp() WHERE notified = false")
        print("Email sent; alerts marked as notified.")

In [0]:
%sql
SELECT notified, risk_band, count(*) AS n FROM pa_clock_dev.gold.gold_pa_alerts GROUP BY 1, 2 ORDER BY 1, 2;